# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is. The functions below
are the ones behind the Monitoring page of the app.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

volatility_flag_success completed


volatility_flag_lookahead completed
arima_forecast_block blocked
prompt_injection_block blocked
frequency_mismatch_block blocked


In [3]:
from quantlab.monitoring import (
    capability_counts,
    guardrail_counts,
    kpis,
    read_events,
    recent_errors,
    stage_latency,
)

events = read_events(settings.events_path)
print(len(events), "events")
pd.Series(kpis(events))

176 events


n_runs               13.0000
n_completed           6.0000
n_blocked             7.0000
n_failed              0.0000
n_in_progress         0.0000
success_rate          0.4615
blocked_rate          0.5385
failed_rate           0.0000
n_replays             2.0000
replay_match_rate     1.0000
cache_hit_rate           NaN
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds):

In [4]:
pd.DataFrame(stage_latency(events))

,stage,count,mean_ms,p95_ms
0,planning,13,0.0,0.0
1,validation,21,0.1,0.2
2,execution,6,114.2,127.9
3,narration,6,0.0,0.0


Which capabilities run most, and which guardrails fire most:

In [5]:
pd.Series(capability_counts(events), name="executions")

load_series            6
resample               6
compute_returns        6
rolling_volatility     6
flag_regime            6
forward_return         4
compare_groups         4
hac_mean_difference    4
Name: executions, dtype: int64

In [6]:
pd.DataFrame(guardrail_counts(events))

,code,name,severity,count
0,G10,MATERIAL_ASSUMPTION,APPROVAL,30
1,G11,PLANNER_REFUSED,BLOCK,4
2,G09,LOOK_AHEAD,APPROVAL,3
3,G08,FREQUENCY_MISMATCH,BLOCK,2
4,G03,INVALID_PARAM,BLOCK,2


The most recent failures (blocked runs and errors):

In [7]:
pd.DataFrame(recent_errors(events, limit=5))

,timestamp,run_id,event_type,error
0,2026-10-18T06:50:48.673Z,20261018-065048-758c27,run_failed,The plan has blocking issues: G08:s6
1,2026-10-18T06:50:48.633Z,20261018-065048-0a8b96,run_failed,The plan has blocking issues: G11:plan
2,2026-10-18T06:50:48.586Z,20261018-065048-a3408c,run_failed,The plan has blocking issues: G11:plan
3,2026-10-18T06:50:37.137Z,20261018-065037-e285e9,run_failed,The plan has blocking issues: G03:plan:edit:s4.capability
4,2026-10-18T06:50:37.102Z,20261018-065037-e285e9,run_failed,The plan has blocking issues: G03:s5:quantile
